# India Runs — Candidate Ranker Sandbox
**Senior AI Engineer — Founding Team @ Redrob AI**

This notebook runs the ranking pipeline on a sample of candidates.

## 1. Setup
Clone the repo and install dependencies.

In [ ]:
import sys, subprocess, time
from pathlib import Path

# Clone repo
if not Path('india-rank').exists():
    subprocess.run(['git', 'clone', 'https://github.com/OmKhatavkar25/india-rank.git'], check=True)
%cd india-rank

# Install deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Setup complete')

## 2. Upload candidate data

Upload a **sample** of candidates.jsonl (first 500 rows recommended):
```bash
head -n 500 candidates.jsonl > sample_candidates.jsonl
```

In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = list(uploaded.keys())[0]
print(f'Uploaded {input_path}')

## 3. Run ranking pipeline

In [ ]:
sys.path.insert(0, 'src')
from candidate_ranker.loader import load_candidates
from candidate_ranker.jd_features import extract_features
from candidate_ranker.jd_scoring import score_candidate, generate_reasoning
from candidate_ranker.semantic import compute_semantic_scores

t0 = time.time()
candidates = load_candidates(input_path)
t1 = time.time()
print(f'Loaded {len(candidates)} candidates in {t1-t0:.1f}s')

cache_path = Path('candidate_embeddings.npy')
semantic_scores = compute_semantic_scores(candidates, cache_path)
t2 = time.time()
print(f'Semantic scores computed in {t2-t1:.1f}s')

scored = []
for c in candidates:
    f = extract_features(c)
    f['semantic_score'] = semantic_scores.get(c.candidate_id, 0.0)
    s = score_candidate(f)
    scored.append((s, c.candidate_id, f, c))

scored.sort(key=lambda x: (-x[0], x[1]))
t3 = time.time()
print(f'Scored {len(scored)} candidates in {t3-t2:.1f}s')
print(f'Total runtime: {t3-t0:.1f}s')

## 4. Results

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for i, (s, cid, f, c) in enumerate(scored[:20]):
    reasoning = generate_reasoning(cid, f, round(s, 4), i + 1, c)
    rows.append({'Rank': i+1, 'Candidate': cid, 'Score': round(s, 4), 'Reasoning': reasoning})

df = pd.DataFrame(rows)
display(df.style.set_properties(**{'text-align': 'left'}))

## 5. Export submission.csv

In [ ]:
from candidate_ranker.exporter import export_submission

ranked = []
for i, (s, cid, f, c) in enumerate(scored[:100]):
    ranked.append({
        'candidate_id': cid,
        'rank': i + 1,
        'score': round(s, 4),
        'reasoning': generate_reasoning(cid, f, round(s, 4), i + 1, c),
    })

out = export_submission(ranked, 'submission.csv')
print(f'Exported to {out}')

from google.colab import files as colab_files
colab_files.download('submission.csv')